# Test python verification packages with real dataset

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
#pip install bomwater

In [3]:
#help(gg.gauge_getter)

In [4]:
#!pip install -e /datasets/work/lw-hydrofct/work/common/Software/xarray_utilities/

In [5]:
import sys,glob
print("Python version")
print (sys.version)
#print("Version info.")
#print (sys.version_info)

Python version
3.10.11 | packaged by conda-forge | (main, May 10 2023, 18:58:44) [GCC 11.3.0]


### Import Various Necessary Libraries Dask and Related Libraries

In [6]:
from dask_jobqueue import SLURMCluster
from dask.distributed import Client
import dask
import os
import uuid

In [7]:
os.environ['USE_PYGEOS'] = '0'
import random
import datetime as dt
import pytz
#from decimal import *#
import geopandas as gpd 
import fiona 
import numpy as np
import math
import pandas as pd 
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.lines as mlines
import xarray as xr
from PIL import Image
import mdba_gauge_getter as gg
from matplotlib.collections import LineCollection
from matplotlib.colors import ListedColormap, BoundaryNorm
import seaborn as sns

In [8]:
#!pip install seaborn --upgrade

### Add proxy binaries to path

In [9]:
pwd = !echo ${PWD}

In [10]:
pwd

['/datasets/work/d61-coastal-forecasting-wp3/work/shr015/Code/python_singularity/swift_notebooks/fcst_verf/Hawkesbury']

### Extend the local python paths with some network drives

In [11]:
extension_python_paths = [ os.environ["HOME"] + '/lib/python3.10/site-packages']
[sys.path.append(an_ext) for an_ext in extension_python_paths]

[None]

### Specify a python exe used by SLURM to create the dask workers


In [12]:
containered_python_exe = f"srun --export=ALL -n $SLURM_NTASKS -c $SLURM_CPUS_PER_TASK   singularity run {os.environ['SINGULARITY_CONTAINER']} python"

### If you run this cell after creating a cluster it will close that cluster 

In [13]:
try:
    cluster.close()
except:
    pass

### Create a cluster
env_extra sets the worker specific environment parameters   

PYTHONPATH is set to include the extension ptyhon paths and the SINGULARITY_BINDPATH bind paths of this jupyter environment for passing to the workers

In [14]:
job_suffix = os.environ['JOB_SUFFIX'] if 'JOB_SUFFIX' in os.environ.keys() else str(uuid.uuid4())[:8]

In [15]:
job_suffix

'd64ff0aa'

## List available project codes

In [16]:
!get_project_codes

/bin/bash: line 1: get_project_codes: command not found


In [17]:
defined = 'NC_IN_GLOB' in os.environ.keys()
if not defined:
    print("WARNING, project code note defined defaulting")
    project_code = 'OD-230112'
else:
    project_code = os.environ['NC_IN_GLOB']

WARNING, project code note defined defaulting


In [18]:
project_code

'OD-230112'

In [19]:
job_extra = f'--account {project_code}'

In [20]:
process_number = 2

In [21]:
cluster = SLURMCluster(
    cores=2, memory="12G", processes=process_number,
    walltime="02:59:00",
    interface='ib0',
    death_timeout=480,
    job_name = f'dask-worker-{job_suffix}',
    job_extra_directives = [job_extra],
    job_script_prologue=[
              'module load singularity', # ensure singularity is loaded
              'export PYTHONPATH=' + ':'.join(extension_python_paths),
              'export SINGULARITY_BINDPATH=' + os.environ['SINGULARITY_BIND'], 
              'export SINGULARITYENV_PREPEND_PATH='+ str(pwd[0]) +':' + str(pwd[0]) + ',/srv/conda/envs/notebook/bin:/srv/conda/condabin:/srv/conda/bin'],
    worker_extra_args = [f'--local-directory={os.environ["SCRATCH3DIR"]}'],
    log_directory = '/home/shr015/singularity_slurm_log_directory',
    python=containered_python_exe,  # use pyhton in container
)

# Debug by running cluster.job_script()

INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at: tcp://10.150.202.155:42375
INFO:distributed.scheduler:  dashboard at:  http://10.150.202.155:8787/status


In [22]:
#cluster.job_script()

### Create a client, this will inject dask into xarray and the distributed cluster into dask

In [23]:
client = Client(cluster, timeout=240)
display(client)

INFO:distributed.scheduler:Receive client connection: Client-66448a1e-6a69-11ef-941b-70b5e8f03b1a
INFO:distributed.core:Starting established connection to tcp://10.150.202.155:48658


Connection method: Cluster object,Cluster type: dask_jobqueue.SLURMCluster
Dashboard: http://10.150.202.155:8787/status,
Dashboard: http://10.150.202.155:8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://10.150.202.155:42375,Workers: 0
Dashboard: http://10.150.202.155:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [24]:
port = client.dashboard_link.split('/')[-2].split(':')[-1]
print(f"Try http://localhost:8888/proxy/{port}/status for the dask dashboard")

Try http://localhost:8888/proxy/8787/status for the dask dashboard


### Scale your workers

With the default config in SLURMCluster above each job will get create 2 workers, one per process and each will have 30gb ram and 4 cores

In [25]:
max_workers = 5

import time
for i in range(0, max_workers): 
    cluster.scale(jobs=i) #yes this looks weird requesting n+1 workers everytime but really it only requests 1 new worker each time
    time.sleep(5)

timeout = 600   # seconds till timeout, timeout if cluster not up in 10 minutes
timeout_start = time.time()
while len(client.ncores().keys())*process_number < max_workers -1:
    if (time.time() > timeout_start+timeout):
        raise Exception(f"Failed to start enough workers in {timeout} seconds, {len(cluster.workers)} started")
    time.sleep(1)

INFO:distributed.scheduler:Register worker <WorkerState 'tcp://10.150.201.203:39831', name: SLURMCluster-0-1, status: init, memory: 0, processing: 0>
INFO:distributed.scheduler:Starting worker compute stream, tcp://10.150.201.203:39831
INFO:distributed.core:Starting established connection to tcp://10.150.201.203:53560
INFO:distributed.scheduler:Register worker <WorkerState 'tcp://10.150.201.203:39251', name: SLURMCluster-0-0, status: init, memory: 0, processing: 0>
INFO:distributed.scheduler:Starting worker compute stream, tcp://10.150.201.203:39251
INFO:distributed.core:Starting established connection to tcp://10.150.201.203:53548
INFO:distributed.scheduler:Register worker <WorkerState 'tcp://10.150.201.17:38043', name: SLURMCluster-3-0, status: init, memory: 0, processing: 0>
INFO:distributed.scheduler:Starting worker compute stream, tcp://10.150.201.17:38043
INFO:distributed.core:Starting established connection to tcp://10.150.201.17:38802
INFO:distributed.scheduler:Register worker 

In [26]:
len(client.ncores())

8

In [27]:
#%load_ext line_profiler

# Petrichor setting
TO run this notebook in petrichor
 - Uncomment out following until "EASI-hub setting"
 - Comment out from "EASI-hub setting" until "End of setting"

In [28]:
#sys.path.append('/datasets/work/lw-hydrofct/work/common/Software/gcm-analysis/scripts')
sys.path.append('/datasets/work/lw-hydrofct/work/common/Software/python_functions/')
import common_functions as cf,plot_utils
from netcdf_utility import nc_utils  #'/datasets/work/lw-hydrofct/work/common/Software/python_functions/netcdf_utility
sys.path.append('/datasets/work/lw-hydrofct/work/common/Software/python_verification/')
import vrf_scores
import crps_class

In [29]:
client.upload_file('/datasets/work/lw-hydrofct/work/common/Software/python_functions/netcdf_utility/nc_utils.py');
client.upload_file('/datasets/work/lw-hydrofct/work/common/Software/python_functions/common_functions.py');
client.upload_file('/datasets/work/lw-hydrofct/work/common/Software/python_verification/vrf_scores.py');

In [30]:
#sys.path.append('/datasets/work/lw-hydrofct/work/common/Software/bjp-g/src')
#import pybjp

In [31]:
#sys.path.append('/datasets/work/lw-hydrofct/work/common/Projects/DPE_Headroom/Code/Matlab/demandsFcts/2Code/f2_waterDemandsBjp')
#import bjpmodel

## Installing libraries
Installing libraries will install to singularity/.local inside the singularity environment this appears at $HOME/.local.

It isn't possible to directly install libraries on hpc nodes apart from interactive and login and in the future it may not be possible to install libraries even on these nodes. To bypass this we use pip download to download libraries and dependencies as whl files to a network drive accessible to HPC machines.

For example we might want to install the library nltk for natural language processing. We have a shared path at /datasets/work/oa-sle/work/python_envs/ben_nglp_1/ accessible to ppts-login. Via windows we can execute a script to download the packages. The cell below is an example script generator for windows downloads. The variable values can be changed to generate a script customized for particular packages.

In [32]:
package_to_install = ''#'dea_tools'#'line_profiler' # insert package to install here e.g. "openpyxl"
my_library_path = "/datasets/work/lw-werp-cafs/work/shr015/Code/python_singularity_envs"
my_ident = "shr015"
container = "/datasets/work/lw-wa-bpa/work/WORKING/ENVS/singularity/pangeo-latest.sif"
windows_script = f"""ssh {my_ident}@petrichor-login.hpc.csiro.au "module load singularity && export SINGULARITY_BINDPATH={my_library_path}:{my_library_path} && export CONTAINER={container} && singularity run {container} /bin/bash -c 'cd {my_library_path}  && /srv/conda/envs/notebook/bin/pip download {package_to_install}'"""

In [33]:
if package_to_install != '':
    print(windows_script)

In [34]:
#%%sh
#pip3 install --find-links /datasets/work/lw-werp-cafs/work/shr015/Code/python_singularity_envs --no-index /datasets/work/lw-werp-cafs/work/shr015/Code/python_singularity_envs/dea_tools-0.2.7-py3-none-any.whl

# EASI-hub setting
TO run this notebook in EASI-hub
 - Comment out from "Petrichor setting" until "EASI-hub setting"
 - Uncomment out following until "End of setting"

In [35]:
#sys.path.extend(['/home/jovyan/nbic-workflow', '/home/jovyan/xarray_utilities','/home/jovyan/climate_change_bias_correction'])

In [36]:
# import xrutils
# import nbic_utils
# import utils
# import delta_correction as wc
# import s3fs

## Cluster

In [37]:
# client, cluster = nbic_utils.easi_cluster(number_of_workers=10)
# client

### End of setting

# Now add you usual codes from here

In [38]:
tstart = time.time()
print('Started: ', time.ctime(tstart))

Started:  Wed Sep  4 12:58:20 2024


In [39]:
dask.config.set(
    {"array.slicing.split_large_chunks": False } #True}
)  # to avoid creating the large chunk in the first place

In [40]:
def round_list(vals,n):
    return list(map(lambda x: round(x, n), vals))

## Some settings

In [41]:
chunking = {'latitude':5,'longitude':5,'time':-1}
chunking = {'time':10,'station':-1,'lead_time':-1,'ens_member':-1}

In [42]:
rootDir = '/datasets/work/d61-coastal-forecasting-wp3/work/shr015'
ecmwf_dir = '/datasets/work/lw-hydrofct/work/common/Projects/Hydro_Tasmania/2021_2022_Ens_inflows_predictions/Data/ecmwf/ens_nwp_nc'
subarea_loc = '/datasets/work/lw-hydrofct/work/common/Projects/Aquawatch/CHyPP/Catchments'
gis_dir = '/datasets/work/d61-coastal-forecasting-wp3/work/shr015/Data/GIS'
sim_dir = '/datasets/work/d61-coastal-forecasting-wp3/work/shr015/SWIFT/Modelling/Sim'
verf_dir =  os.path.join(rootDir,'CHyPP','Verification')
fig_dir =  os.path.join(rootDir,'CHyPP','Verification')

In [43]:
catchment = 'Hawkesbury'
raw_fcst_name = 'ECMWF00Z_ens'
run_name = 'rpp_ECMWF00Z_ensmean'
output_filename_prefix = 'RPP00Z_1' 
timestep = 24  # hour

In [44]:
plt.rcParams['figure.figsize'] = (15,8) # Note that matplotlib.rcParams == plt.rcParams
plt.rcParams['xtick.labelsize']= 10 
plt.rcParams['ytick.labelsize'] = 10   # plt.rc('ytick', labelsize=14) 
plt.rc('axes', labelsize=12, titlesize=12)
plt.rc('legend',fontsize=10)
#figsize = (15, 11.25)

In [45]:
proj = {'WGS84':"EPSG:4326",  #project to WGS 84 lat and lon, geographic
       'Mercator':"EPSG:3395", # 'Mercator
        'Albers':"EPSG:3577",  # GDA94) as its horizontal datum and the Albers Equal Area Conic projection for its map projection
       'GDA94':"EPSG:4283"}   # GDA94  lat and lon

In [46]:
#gauge = '130406A'

In [47]:
start_date = '2012-04-05'
end_date = '2022-07-06' # This is obs rain and pet, 
end_date_ecmwf = '2023-05-31'
end_date_flow = '2023-09-19'
days = pd.date_range(start_date,end_date)
start_date_obj = dt.datetime.strptime(start_date, '%Y-%m-%d')
end_date_obj = dt.datetime.strptime(end_date, '%Y-%m-%d')

In [48]:
verf_start = '2012-04-05T23:00:00'
verf_end = '2022-07-03T23:00:00'
verf_period = pd.date_range(verf_start,verf_end)
verf_period

DatetimeIndex(['2012-04-05 23:00:00', '2012-04-06 23:00:00',
               '2012-04-07 23:00:00', '2012-04-08 23:00:00',
               '2012-04-09 23:00:00', '2012-04-10 23:00:00',
               '2012-04-11 23:00:00', '2012-04-12 23:00:00',
               '2012-04-13 23:00:00', '2012-04-14 23:00:00',
               ...
               '2022-06-24 23:00:00', '2022-06-25 23:00:00',
               '2022-06-26 23:00:00', '2022-06-27 23:00:00',
               '2022-06-28 23:00:00', '2022-06-29 23:00:00',
               '2022-06-30 23:00:00', '2022-07-01 23:00:00',
               '2022-07-02 23:00:00', '2022-07-03 23:00:00'],
              dtype='datetime64[ns]', length=3742, freq='D')

In [49]:
cmap = plt.get_cmap("tab10")
col  = cmap.colors
col

((0.12156862745098039, 0.4666666666666667, 0.7058823529411765),
 (1.0, 0.4980392156862745, 0.054901960784313725),
 (0.17254901960784313, 0.6274509803921569, 0.17254901960784313),
 (0.8392156862745098, 0.15294117647058825, 0.1568627450980392),
 (0.5803921568627451, 0.403921568627451, 0.7411764705882353),
 (0.5490196078431373, 0.33725490196078434, 0.29411764705882354),
 (0.8901960784313725, 0.4666666666666667, 0.7607843137254902),
 (0.4980392156862745, 0.4980392156862745, 0.4980392156862745),
 (0.7372549019607844, 0.7411764705882353, 0.13333333333333333),
 (0.09019607843137255, 0.7450980392156863, 0.8117647058823529))

In [50]:
sns_color = sns.color_palette()
sns_color

[(0.12156862745098039, 0.4666666666666667, 0.7058823529411765),
 (1.0, 0.4980392156862745, 0.054901960784313725),
 (0.17254901960784313, 0.6274509803921569, 0.17254901960784313),
 (0.8392156862745098, 0.15294117647058825, 0.1568627450980392),
 (0.5803921568627451, 0.403921568627451, 0.7411764705882353),
 (0.5490196078431373, 0.33725490196078434, 0.29411764705882354),
 (0.8901960784313725, 0.4666666666666667, 0.7607843137254902),
 (0.4980392156862745, 0.4980392156862745, 0.4980392156862745),
 (0.7372549019607844, 0.7411764705882353, 0.13333333333333333),
 (0.09019607843137255, 0.7450980392156863, 0.8117647058823529)]

In [51]:
sns.color_palette("tab10")

[(0.12156862745098039, 0.4666666666666667, 0.7058823529411765),
 (1.0, 0.4980392156862745, 0.054901960784313725),
 (0.17254901960784313, 0.6274509803921569, 0.17254901960784313),
 (0.8392156862745098, 0.15294117647058825, 0.1568627450980392),
 (0.5803921568627451, 0.403921568627451, 0.7411764705882353),
 (0.5490196078431373, 0.33725490196078434, 0.29411764705882354),
 (0.8901960784313725, 0.4666666666666667, 0.7607843137254902),
 (0.4980392156862745, 0.4980392156862745, 0.4980392156862745),
 (0.7372549019607844, 0.7411764705882353, 0.13333333333333333),
 (0.09019607843137255, 0.7450980392156863, 0.8117647058823529)]

In [52]:
cumulative = False

### SWIFT Model setting

In [53]:
calObjs=['7DAY']    #%;'KGE'; 'NSEBIAS'};
rrModel = 'GR4J' #%{'GR4J';'PDM'};
routMod = 'LAR'
ecModels = ['NoErrorCorrection','MAERRIS']
frcngs = ['RPP00Z'];#['obs', 'ECMWF00Z_ensmean', 'ECMWF00Z_ens', 'RPP00Z'];
cv_yrs= np.arange(start_date_obj.year,end_date_obj.year+1)
cv_yrs

array([2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022])

In [54]:
metrics = ['NSE','KGE','CORR','RMSE','MAE','BIAS','MEAN','MEDIAN'];
s2Window = 10;
stage = 2
error_corrected_perf = True
ecNodes = None 
insertionNodes = '514'

## Helper functions

In [55]:
def gen_suffix(ecModel,restType=1,s2Window=10,ecNodes=[],insertionNodes=[],insertion_type='climMedian',more_suffix=''):
    outSuffix = ''   
    if not ecModel == "NoErrorCorrection":     
    	suffix=f"_s2W{s2Window}"
    else:
    	suffix = ''
    	restType = 0
    if not restType == 0:
    	outSuffix = f"_RstrTyp{restType}"
    if len(ecNodes) > 0:
    	suffix = f"{suffix}_ecNodes_{','.join(map(str, ecNodes))}"
    if len(insertionNodes) > 0:
    	suffix = f"{suffix}_insertionNodes_{','.join(map(str, insertionNodes))}"
    
    outSuffix = f"{outSuffix}_{insertion_type}Insertion{more_suffix}"
    return suffix,outSuffix        

In [56]:
def read_fcst(rootDir,catchment,rrModel,routMod,ecModel,calObj,frcng,suffix='',outSuffix='',monthly_output=False):
    out_nc_dir =  os.path.join(rootDir,'SWIFT','Modelling','Fct',catchment,ecModel,calObj,frcng)
    ds_all = []
    for yr in cv_yrs:    
        if monthly_output:
            if yr==2023:
                cv_months = range(1,6) 
            else:
                cv_months = range(1,13) 
            for month in cv_months: 
                #print(month)
                filename = f"{catchment}_{rrModel}{routMod}{ecModel}_{calObj}_fct_xv{yr}_m{month:02d}_{frcng}frcng{suffix}{outSuffix}.nc"
                out_nc_file =os.path.join(out_nc_dir,filename)
                print(f"Reading {out_nc_file}")
                ds = cf.open_dataset_decode_time(out_nc_file)
                ds_all.append(ds)
        else:
            filename = f"{catchment}_{rrModel}{routMod}{ecModel}_{calObj}_fct_xv{yr}_{frcng}frcng{suffix}{outSuffix}.nc"      
            out_nc_file =os.path.join(out_nc_dir,filename)
            print(f"Reading {out_nc_file}")
            ds = cf.open_dataset_decode_time(out_nc_file)
            ds_all.append(ds)
    fcst = xr.concat(ds_all, dim='time',data_vars=['q_sim'])
    station = fcst['station'].values
    id = fcst['station_id'].values
    return fcst,station,id

In [57]:
sns_color

[(0.12156862745098039, 0.4666666666666667, 0.7058823529411765),
 (1.0, 0.4980392156862745, 0.054901960784313725),
 (0.17254901960784313, 0.6274509803921569, 0.17254901960784313),
 (0.8392156862745098, 0.15294117647058825, 0.1568627450980392),
 (0.5803921568627451, 0.403921568627451, 0.7411764705882353),
 (0.5490196078431373, 0.33725490196078434, 0.29411764705882354),
 (0.8901960784313725, 0.4666666666666667, 0.7607843137254902),
 (0.4980392156862745, 0.4980392156862745, 0.4980392156862745),
 (0.7372549019607844, 0.7411764705882353, 0.13333333333333333),
 (0.09019607843137255, 0.7450980392156863, 0.8117647058823529)]

### Load calibration gauge sites

In [58]:
streamflow_gauges_file = os.path.join(rootDir,'Data','Flow',catchment,'Calibration_gauge_sites.csv')

In [59]:
streamflow_gauges_df = pd.read_csv(streamflow_gauges_file)
streamflow_gauges_df

,GaugeID,ID,CalNode,Calib(Y/N),SimOption,River Name,Latitude,Longitude,Area_km2,Comment
0,212018,1,104,1,1,Capertee River at Glen Davis,-33.121000,150.280600,1010,BoM online data (1970/08/15 to 2024/5/29)
1,212228,2,607,1,1,Macdonal River at St Albans,-33.290000,150.970000,-9999,BoM online data (1970/07/19 to 2024/5/06)
2,212290,3,611,1,1,Colo River at Upper Colo,-33.418320,150.725890,-9999,NSW data 1975-02-07 2024-02-12 as BoM data 201...
3,212021,4,207,1,1,Macdonald River at Howes Valley,-32.861096,150.810589,299,BoM online data 1976-02-10 2024-05-29
4,212200,5,505,0,1,Hawkesbury River at North Richmond,-33.589301,150.714031,-9999,Most of BoM or NSW data is missing so converte...
5,212201,6,512,1,1,Nepean River at Penrith,-33.746778,150.682500,11000,NSW 1968-12-21 2024-02-12
6,212320,7,511,1,1,South Creek @ Elisabeth Drive,-33.877350,150.768594,88,BoM 1970-06-01 2024-05-29
7,212297,8,515,1,1,South Creek at Richmond Road,-33.680000,150.810000,-9999,BoM 1982-03-06 2024-05-29
8,212296,9,516,1,1,Eastern Creek at Riverstone,-33.680000,150.850000,-9999,BoM 1982-10-16 2024-05-29
9,212048,10,518,1,1,South Creek at Great Western Highway,-33.769400,150.761844,250,BoM 1986-02-25 2024-05-29


### Load observed flow

In [60]:
out_nc_file =  os.path.join(rootDir,'SWIFT','Modelling','Data',catchment,f"{catchment}_flow_da.nc")
obs = cf.open_dataset_decode_time(out_nc_file)
obs

<xarray.Dataset> Size: 3MB
Dimensions:           (station: 16, lead_time: 1, ens_member: 1, time: 22817)
Coordinates:
  * station           (station) float64 128B 1.0 2.0 3.0 4.0 ... 14.0 15.0 16.0
  * lead_time         (lead_time) float64 8B 0.0
  * ens_member        (ens_member) float64 8B 1.0
  * time              (time) datetime64[ns] 183kB 1962-01-04T23:00:00 ... 202...
Data variables:
    station_id        (station) float64 128B ...
    station_name      (station) |S30 480B ...
    other_station_id  (station) |S30 480B ...
    lat               (station) float32 64B ...
    lon               (station) float32 64B ...
    area              (station) float32 64B ...
    q_obs             (time, ens_member, station, lead_time) float32 1MB ...
    q_obs_qual        (time, ens_member, station, lead_time) float32 1MB ...
Attributes:
    history:                 Created 2024-07-05 09:19:56
    title:                   
    institution:             CSIRO Environment
    source:                  /datasets/work/d61-coastal-forecasting-wp3/work/...
    catchment:               Hawkesbury
    STF_convention_version:  2
    STF_nc_spec:             https://wiki.csiro.au/display/wirada/NetCDF+for+...
    comment:                 created from csv_streamflow_to_netcdf_final_Hawk...

In [61]:
station_obs = obs['station'].values
id_obs = obs['station_id'].values
print(station_obs)
print(id_obs)

[ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15. 16.]
[104. 607. 207. 511. 515. 516. 518. 611. 512. 505. 513. 609. 517. 603.
 605. 514.]


In [62]:
ecNodes = [] #[46,42,66,72]
insertionNodes=  ['514']
insertion_type = 'climMedian'
ci = [0.05,0.95]  # uncertainty bounds
#gauge=46

In [63]:
fig_dir = os.path.join(rootDir,'SWIFT','Verification','QFcst',catchment)
fig_dir

'/datasets/work/d61-coastal-forecasting-wp3/work/shr015/SWIFT/Verification/QFcst/Hawkesbury'

### Load climatology

In [64]:
out_nc_file =  os.path.join(rootDir,'SWIFT','Modelling','Data',catchment,f"{catchment}_xval_clim_flow_da_from1970.nc")
clim = cf.open_dataset_decode_time(out_nc_file)
clim

<xarray.Dataset> Size: 36MB
Dimensions:       (station: 16, lead_time: 3, ens_member: 50, time: 3745)
Coordinates:
  * station       (station) int32 64B 104 607 207 511 515 ... 517 603 605 514
  * lead_time     (lead_time) int32 12B 1 2 3
  * ens_member    (ens_member) int32 200B 1 2 3 4 5 6 7 ... 44 45 46 47 48 49 50
  * time          (time) datetime64[ns] 30kB 2012-04-05T23:00:00 ... 2022-07-...
Data variables:
    station_id    (station) int32 64B ...
    station_name  (station) |S30 480B ...
    lat           (station) float32 64B ...
    lon           (station) float32 64B ...
    area          (station) float32 64B ...
    q_sim         (time, ens_member, station, lead_time) float32 36MB ...
Attributes:
    title:                   Climatology Streamflow
    institution:             CSIRO Land & Water
    source:                  
    catchment:               Hawkesbury
    STF_convention_version:  2.0
    STF_nc_spec:             https://wiki.csiro.au/display/wirada/NetCDF+for+...
    comment:                 cross-validate computed for g:\work\shr015\SWIFT...
    history:                 2024-08-05 11:14:40 +10.0 - File created

In [65]:
fcst_all = [clim]
st_all = [clim['station'].values]
st_id_all = [clim['station_id'].values]

In [66]:
st_id_all

[array([104, 607, 207, 511, 515, 516, 518, 611, 512, 505, 513, 609, 517,
        603, 605, 514], dtype=int32)]

In [67]:
suffix,outSuffix = gen_suffix(ecModels[1],ecNodes=ecNodes,insertionNodes=insertionNodes,insertion_type=insertion_type,more_suffix='_from1970')
#outSuffix = f"_RstrTyp1{outSuffix}"
suffix,outSuffix 

('_s2W10_insertionNodes_514', '_RstrTyp1_climMedianInsertion_from1970')

In [68]:
for frcng in frcngs:
    fcst,st,st_id = read_fcst(rootDir,catchment,rrModel,routMod, ecModels[1],calObjs[0],frcng,monthly_output=False,suffix=suffix,outSuffix=outSuffix)  
    fcst_all.append(fcst)
    st_all.append(st)
    st_id_all.append(st_id)

Reading /datasets/work/d61-coastal-forecasting-wp3/work/shr015/SWIFT/Modelling/Fct/Hawkesbury/MAERRIS/7DAY/RPP00Z/Hawkesbury_GR4JLARMAERRIS_7DAY_fct_xv2012_RPP00Zfrcng_s2W10_insertionNodes_514_RstrTyp1_climMedianInsertion_from1970.nc
Reading /datasets/work/d61-coastal-forecasting-wp3/work/shr015/SWIFT/Modelling/Fct/Hawkesbury/MAERRIS/7DAY/RPP00Z/Hawkesbury_GR4JLARMAERRIS_7DAY_fct_xv2013_RPP00Zfrcng_s2W10_insertionNodes_514_RstrTyp1_climMedianInsertion_from1970.nc
Reading /datasets/work/d61-coastal-forecasting-wp3/work/shr015/SWIFT/Modelling/Fct/Hawkesbury/MAERRIS/7DAY/RPP00Z/Hawkesbury_GR4JLARMAERRIS_7DAY_fct_xv2014_RPP00Zfrcng_s2W10_insertionNodes_514_RstrTyp1_climMedianInsertion_from1970.nc
Reading /datasets/work/d61-coastal-forecasting-wp3/work/shr015/SWIFT/Modelling/Fct/Hawkesbury/MAERRIS/7DAY/RPP00Z/Hawkesbury_GR4JLARMAERRIS_7DAY_fct_xv2015_RPP00Zfrcng_s2W10_insertionNodes_514_RstrTyp1_climMedianInsertion_from1970.nc
Reading /datasets/work/d61-coastal-forecasting-wp3/work/shr015/S

### Time series plot

In [69]:
fcst_name_list = ['CLim','CHyPP']#['CLim','Obs','ECMWF_ensmean','ECMWF_ens','CHyPP']

## Outlet

In [70]:
gauge = 609

In [71]:
river_name = streamflow_gauges_df[streamflow_gauges_df['CalNode']==gauge]['River Name'].values[0]
gauge_id = streamflow_gauges_df[streamflow_gauges_df['CalNode']==gauge]['GaugeID'].values[0]
gauge_name = f"{river_name} {gauge_id}"

In [72]:
#nse_out,bias_out,crps_out,pit_alpha_out = plot_utils.plot_fcst_verif(obs,fcst_all,st_id_all,st_all,gauge,verf_period,ci=ci,fcst_name_list=fcst_name_list,gauge_name=gauge_name,nrow_legend=2)

## Use score package

In [73]:
from scores.probability import crps_for_ensemble

In [74]:
obs

<xarray.Dataset> Size: 3MB
Dimensions:           (station: 16, lead_time: 1, ens_member: 1, time: 22817)
Coordinates:
  * station           (station) float64 128B 1.0 2.0 3.0 4.0 ... 14.0 15.0 16.0
  * lead_time         (lead_time) float64 8B 0.0
  * ens_member        (ens_member) float64 8B 1.0
  * time              (time) datetime64[ns] 183kB 1962-01-04T23:00:00 ... 202...
Data variables:
    station_id        (station) float64 128B 104.0 607.0 207.0 ... 605.0 514.0
    station_name      (station) |S30 480B ...
    other_station_id  (station) |S30 480B ...
    lat               (station) float32 64B ...
    lon               (station) float32 64B ...
    area              (station) float32 64B ...
    q_obs             (time, ens_member, station, lead_time) float32 1MB ...
    q_obs_qual        (time, ens_member, station, lead_time) float32 1MB ...
Attributes:
    history:                 Created 2024-07-05 09:19:56
    title:                   
    institution:             CSIRO Environment
    source:                  /datasets/work/d61-coastal-forecasting-wp3/work/...
    catchment:               Hawkesbury
    STF_convention_version:  2
    STF_nc_spec:             https://wiki.csiro.au/display/wirada/NetCDF+for+...
    comment:                 created from csv_streamflow_to_netcdf_final_Hawk...

In [75]:
fcst_all[0]

<xarray.Dataset> Size: 36MB
Dimensions:       (station: 16, lead_time: 3, ens_member: 50, time: 3745)
Coordinates:
  * station       (station) int32 64B 104 607 207 511 515 ... 517 603 605 514
  * lead_time     (lead_time) int32 12B 1 2 3
  * ens_member    (ens_member) int32 200B 1 2 3 4 5 6 7 ... 44 45 46 47 48 49 50
  * time          (time) datetime64[ns] 30kB 2012-04-05T23:00:00 ... 2022-07-...
Data variables:
    station_id    (station) int32 64B 104 607 207 511 515 ... 517 603 605 514
    station_name  (station) |S30 480B ...
    lat           (station) float32 64B ...
    lon           (station) float32 64B ...
    area          (station) float32 64B ...
    q_sim         (time, ens_member, station, lead_time) float32 36MB ...
Attributes:
    title:                   Climatology Streamflow
    institution:             CSIRO Land & Water
    source:                  
    catchment:               Hawkesbury
    STF_convention_version:  2.0
    STF_nc_spec:             https://wiki.csiro.au/display/wirada/NetCDF+for+...
    comment:                 cross-validate computed for g:\work\shr015\SWIFT...
    history:                 2024-08-05 11:14:40 +10.0 - File created

In [76]:
obs1 = obs.squeeze("ens_member")
obs1

<xarray.Dataset> Size: 3MB
Dimensions:           (station: 16, lead_time: 1, time: 22817)
Coordinates:
  * station           (station) float64 128B 1.0 2.0 3.0 4.0 ... 14.0 15.0 16.0
  * lead_time         (lead_time) float64 8B 0.0
    ens_member        float64 8B 1.0
  * time              (time) datetime64[ns] 183kB 1962-01-04T23:00:00 ... 202...
Data variables:
    station_id        (station) float64 128B 104.0 607.0 207.0 ... 605.0 514.0
    station_name      (station) |S30 480B ...
    other_station_id  (station) |S30 480B ...
    lat               (station) float32 64B ...
    lon               (station) float32 64B ...
    area              (station) float32 64B ...
    q_obs             (time, station, lead_time) float32 1MB ...
    q_obs_qual        (time, station, lead_time) float32 1MB ...
Attributes:
    history:                 Created 2024-07-05 09:19:56
    title:                   
    institution:             CSIRO Environment
    source:                  /datasets/work/d61-coastal-forecasting-wp3/work/...
    catchment:               Hawkesbury
    STF_convention_version:  2
    STF_nc_spec:             https://wiki.csiro.au/display/wirada/NetCDF+for+...
    comment:                 created from csv_streamflow_to_netcdf_final_Hawk...

In [77]:
obs1['station'] = fcst_all[0]['station']
obs1

<xarray.Dataset> Size: 3MB
Dimensions:           (station: 16, lead_time: 1, time: 22817)
Coordinates:
  * station           (station) int32 64B 104 607 207 511 ... 517 603 605 514
  * lead_time         (lead_time) float64 8B 0.0
    ens_member        float64 8B 1.0
  * time              (time) datetime64[ns] 183kB 1962-01-04T23:00:00 ... 202...
Data variables:
    station_id        (station) float64 128B 104.0 607.0 207.0 ... 605.0 514.0
    station_name      (station) |S30 480B ...
    other_station_id  (station) |S30 480B ...
    lat               (station) float32 64B ...
    lon               (station) float32 64B ...
    area              (station) float32 64B ...
    q_obs             (time, station, lead_time) float32 1MB ...
    q_obs_qual        (time, station, lead_time) float32 1MB ...
Attributes:
    history:                 Created 2024-07-05 09:19:56
    title:                   
    institution:             CSIRO Environment
    source:                  /datasets/work/d61-coastal-forecasting-wp3/work/...
    catchment:               Hawkesbury
    STF_convention_version:  2
    STF_nc_spec:             https://wiki.csiro.au/display/wirada/NetCDF+for+...
    comment:                 created from csv_streamflow_to_netcdf_final_Hawk...

In [78]:
fcst1 = fcst_all[0]
fcst1

<xarray.Dataset> Size: 36MB
Dimensions:       (station: 16, lead_time: 3, ens_member: 50, time: 3745)
Coordinates:
  * station       (station) int32 64B 104 607 207 511 515 ... 517 603 605 514
  * lead_time     (lead_time) int32 12B 1 2 3
  * ens_member    (ens_member) int32 200B 1 2 3 4 5 6 7 ... 44 45 46 47 48 49 50
  * time          (time) datetime64[ns] 30kB 2012-04-05T23:00:00 ... 2022-07-...
Data variables:
    station_id    (station) int32 64B 104 607 207 511 515 ... 517 603 605 514
    station_name  (station) |S30 480B ...
    lat           (station) float32 64B ...
    lon           (station) float32 64B ...
    area          (station) float32 64B ...
    q_sim         (time, ens_member, station, lead_time) float32 36MB ...
Attributes:
    title:                   Climatology Streamflow
    institution:             CSIRO Land & Water
    source:                  
    catchment:               Hawkesbury
    STF_convention_version:  2.0
    STF_nc_spec:             https://wiki.csiro.au/display/wirada/NetCDF+for+...
    comment:                 cross-validate computed for g:\work\shr015\SWIFT...
    history:                 2024-08-05 11:14:40 +10.0 - File created

In [79]:
lead_time = [24, 48, 72]
time_deltas = np.timedelta64(lead_time[0], "h")

In [80]:
verf_period

DatetimeIndex(['2012-04-05 23:00:00', '2012-04-06 23:00:00',
               '2012-04-07 23:00:00', '2012-04-08 23:00:00',
               '2012-04-09 23:00:00', '2012-04-10 23:00:00',
               '2012-04-11 23:00:00', '2012-04-12 23:00:00',
               '2012-04-13 23:00:00', '2012-04-14 23:00:00',
               ...
               '2022-06-24 23:00:00', '2022-06-25 23:00:00',
               '2022-06-26 23:00:00', '2022-06-27 23:00:00',
               '2022-06-28 23:00:00', '2022-06-29 23:00:00',
               '2022-06-30 23:00:00', '2022-07-01 23:00:00',
               '2022-07-02 23:00:00', '2022-07-03 23:00:00'],
              dtype='datetime64[ns]', length=3742, freq='D')

In [81]:
target_fcst_dates = (
            fcst1.sel(time=verf_period)["time"].values + time_deltas
        )

In [82]:
target_fcst_dates

array(['2012-04-06T23:00:00.000000000', '2012-04-07T23:00:00.000000000',
       '2012-04-08T23:00:00.000000000', ...,
       '2022-07-02T23:00:00.000000000', '2022-07-03T23:00:00.000000000',
       '2022-07-04T23:00:00.000000000'], dtype='datetime64[ns]')

In [83]:
# obs flow
obs_verf = obs1["q_obs"].sel(time=target_fcst_dates)
obs_verf

<xarray.DataArray 'q_obs' (time: 3742, station: 16, lead_time: 1)> Size: 239kB
[59872 values with dtype=float32]
Coordinates:
  * station     (station) int32 64B 104 607 207 511 515 ... 609 517 603 605 514
  * lead_time   (lead_time) float64 8B 0.0
    ens_member  float64 8B 1.0
  * time        (time) datetime64[ns] 30kB 2012-04-06T23:00:00 ... 2022-07-04...
Attributes:
    standard_name:         q_obs
    long_name:             observed streamflow
    units:                 m3/s
    type:                  3
    type_description:      averaged over the preceding interval
    dat_type:              obs
    dat_type_description:  observed
    location_type:         Point

In [84]:
obs_verf = obs_verf.squeeze("lead_time")
obs_verf

<xarray.DataArray 'q_obs' (time: 3742, station: 16)> Size: 239kB
[59872 values with dtype=float32]
Coordinates:
  * station     (station) int32 64B 104 607 207 511 515 ... 609 517 603 605 514
    lead_time   float64 8B 0.0
    ens_member  float64 8B 1.0
  * time        (time) datetime64[ns] 30kB 2012-04-06T23:00:00 ... 2022-07-04...
Attributes:
    standard_name:         q_obs
    long_name:             observed streamflow
    units:                 m3/s
    type:                  3
    type_description:      averaged over the preceding interval
    dat_type:              obs
    dat_type_description:  observed
    location_type:         Point

In [85]:
fcst_verf = fcst1["q_sim"].sel(lead_time=lead_time[0]/24,time=verf_period) 
fcst_verf             

<xarray.DataArray 'q_sim' (time: 3742, ens_member: 50, station: 16)> Size: 12MB
[2993600 values with dtype=float32]
Coordinates:
  * station     (station) int32 64B 104 607 207 511 515 ... 609 517 603 605 514
    lead_time   int32 4B 1
  * ens_member  (ens_member) int32 200B 1 2 3 4 5 6 7 8 ... 44 45 46 47 48 49 50
  * time        (time) datetime64[ns] 30kB 2012-04-05T23:00:00 ... 2022-07-03...
Attributes:
    standard_name:         q_sim
    long_name:             simulated streamflow
    units:                 m3/s
    type:                  11.0
    type_description:      climatology data - averaged over the preceding int...
    dat_type:              fct
    dat_type_description:  forecast
    location_type:         area

## Export data

In [96]:
obs_verf = obs_verf.drop_vars('lead_time')
obs_verf

<xarray.DataArray 'q_obs' (time: 3742, station: 16)> Size: 239kB
[59872 values with dtype=float32]
Coordinates:
  * station     (station) int32 64B 104 607 207 511 515 ... 609 517 603 605 514
    ens_member  float64 8B 1.0
  * time        (time) datetime64[ns] 30kB 2012-04-06T23:00:00 ... 2022-07-04...
Attributes:
    standard_name:         q_obs
    long_name:             observed streamflow
    units:                 m3/s
    type:                  3
    type_description:      averaged over the preceding interval
    dat_type:              obs
    dat_type_description:  observed
    location_type:         Point

In [97]:
fcst_verf = fcst_verf.drop_vars('lead_time')
fcst_verf

<xarray.DataArray 'q_sim' (time: 3742, ens_member: 50, station: 16)> Size: 12MB
[2993600 values with dtype=float32]
Coordinates:
  * station     (station) int32 64B 104 607 207 511 515 ... 609 517 603 605 514
  * ens_member  (ens_member) int32 200B 1 2 3 4 5 6 7 8 ... 44 45 46 47 48 49 50
  * time        (time) datetime64[ns] 30kB 2012-04-05T23:00:00 ... 2022-07-03...
Attributes:
    standard_name:         q_sim
    long_name:             simulated streamflow
    units:                 m3/s
    type:                  11.0
    type_description:      climatology data - averaged over the preceding int...
    dat_type:              fct
    dat_type_description:  forecast
    location_type:         area

In [98]:
### Export data
# Combine them into a Dataset
stremflow_data = xr.Dataset({
    'obs': obs_verf,
    'fcst': fcst_verf
})
#stremflow_data = xr.merge([obs_verf, fcst_verf], compat='override')
stremflow_data

<xarray.Dataset> Size: 12MB
Dimensions:     (station: 16, time: 3743, ens_member: 50)
Coordinates:
  * station     (station) int32 64B 104 607 207 511 515 ... 609 517 603 605 514
  * time        (time) datetime64[ns] 30kB 2012-04-05T23:00:00 ... 2022-07-04...
  * ens_member  (ens_member) int32 200B 1 2 3 4 5 6 7 8 ... 44 45 46 47 48 49 50
Data variables:
    obs         (time, station) float32 240kB nan nan nan ... nan nan 2.171e+03
    fcst        (time, ens_member, station) float32 12MB 0.009 1.8 ... nan nan

In [100]:
stremflow_data.to_netcdf('stremflow_data_for_scores.nc')

In [101]:
## Load dataset
streamflow = xr.open_dataset('stremflow_data_for_scores.nc')
streamflow

<xarray.Dataset> Size: 12MB
Dimensions:     (station: 16, time: 3743, ens_member: 50)
Coordinates:
  * station     (station) int32 64B 104 607 207 511 515 ... 609 517 603 605 514
  * time        (time) datetime64[ns] 30kB 2012-04-05T23:00:00 ... 2022-07-04...
  * ens_member  (ens_member) int32 200B 1 2 3 4 5 6 7 8 ... 44 45 46 47 48 49 50
Data variables:
    obs         (time, station) float32 240kB ...
    fcst        (time, ens_member, station) float32 12MB ...

## Run scores

In [103]:
import scoringrules
from scores.probability import crps_for_ensemble
import properscoring

In [104]:
def crps_from_empirical_cdf(pred, obs, dim=0):
    #https://docs.nvidia.com/deeplearning/modulus/modulus-core/_modules/modulus/metrics/general/crps.html
    n = pred.shape[dim]
    pred = np.sort(pred, axis=dim)
    ans = np.zeros_like(obs)

    # dx [F(x) - H(x-y)]^2 = dx [0 - 1]^2 = dx
    # val = ensemble[0] - truth
    val = (pred[0, :] - obs)
    #val = (pred[:, 0] - obs)
    ans += np.maximum(val, 0.0)

    for i in range(n - 1):
        x0 = pred[i, :]
        x1 = pred[i+1, :]

        cdf = (i + 1) / n

        # a. case y < x0
        val = (x1 - x0) * (cdf - 1) ** 2
        mask = obs < x0
        ans += val * mask

        # b. case x0 <= y <= x1
        val = (obs - x0) * cdf**2 + (x1 - obs) * (cdf - 1) ** 2
        mask = (obs >= x0) & (obs <= x1)
        ans += val * mask

        # c. case x1 < t
        mask = obs > x1
        val = (x1 - x0) * cdf**2
        ans += val * mask

    # dx [F(x) - H(x-y)]^2 = dx [1 - 0]^2 = dx
    val = obs - pred[-1, :]
    ans += np.maximum(val, 0.0)
    return ans

In [135]:
def energy_score(forecasts, obs):

    # must have dimensions of (num_variables, num_ensembles). Variables can include different locations, time steps, climate variables etc

    num_samples = forecasts.shape[1]

    s1 = np.sqrt(np.sum(np.square(forecasts - obs[:, np.newaxis]), axis=0)).sum()

    pairwise_diffs = forecasts[:, :, np.newaxis] - forecasts[:, np.newaxis, :]
    s2 = np.sqrt(np.sum(np.square(pairwise_diffs), axis=0)).sum()

    es = (s1 / num_samples) - s2 / (2 * num_samples**2)
    
    return es

In [227]:
def energy_score_multiple(f_ens, o,return_mean=True):
    
    #num_samples = f_ens.shape[1]
    shape = f_ens.shape
    # handle nans
    mean_fcst = np.mean(f_ens,axis=1) #update by Durga
    nanindx = ~np.isnan(o) & ~np.isnan(mean_fcst)    
    #print(nanindx)
    o = o[nanindx]
    f_ens = f_ens[nanindx,:]    
    if len(shape) == 2 and shape[0] > 1 and shape[1] > 1:  # check for matrix
        num_events = f_ens.shape[0]
        num_ens = f_ens.shape[1]
        crps = []
        for i in range(num_events):
            #ff = f_ens[i,:].reshape(1,num_ens)
            ff = f_ens[i,:]
            ff = ff[np.newaxis, :]   # should be size of 1 by num_samples  
            oo = o[i]
            oo = np.array([oo])
            crps.append(energy_score(ff, oo))   # obs should be 
        #print(f"Fcst shape {ff.shape}, Obs shape: {oo.shape}")
        if return_mean:
            return np.mean(crps)
        else:
            return np.asarray(crps)
    else: 
        return energy_score(f_ens, o)  

In [120]:
def remove_nans(fcst,obs,site):
    ff = fcst.isel(station=site).values
    oo = obs.isel(station=site).values   
    mean_fcst = np.mean(ff,axis=1) 
    nanindx = ~np.isnan(oo) & ~np.isnan(mean_fcst)
    oo = oo[nanindx]
    ff = ff[nanindx,:] 
    return ff,oo

In [112]:
nsite = streamflow['fcst'].shape[2]
nsite

16

### 1. scores.probability.crps_for_ensemble

In [102]:
%%time
# Specifying method='ecdf' assumes the empirical CDF
scores_crps = crps_for_ensemble(streamflow['fcst'], streamflow['obs'], ensemble_member_dim='ens_member', method='ecdf',preserve_dims='station')

CPU times: user 554 ms, sys: 174 ms, total: 728 ms
Wall time: 734 ms


### 2. scoringrules.crps_ensemble

In [128]:
%%time
ens_mem_dim = streamflow['fcst'].get_axis_num("ens_member")
scoringrules_crps = []
for s in range(nsite):
    fcst,obs = remove_nans(streamflow['fcst'],streamflow['obs'],s)
    scoringrules_crps_vals = scoringrules.crps_ensemble(obs,fcst, axis=1, estimator="nrg")
    scoringrules_crps.append(scoringrules_crps_vals.mean().item())

CPU times: user 189 ms, sys: 61.6 ms, total: 250 ms
Wall time: 245 ms


### 3. vrf_scores.crps_ecdf_multiple

In [130]:
%%time
vrf_scores_crps = []
for s in range(nsite):
    fcst,obs = remove_nans(streamflow['fcst'],streamflow['obs'],s)
    vrf_scores_crps.append(vrf_scores.crps_ecdf_multiple(fcst,obs)) 

CPU times: user 4.51 s, sys: 231 ms, total: 4.75 s
Wall time: 4.58 s


### 4. properscoring.crps_ensemble

In [133]:
%%time
properscoring_crps = []
for s in range(nsite):
    fcst,obs = remove_nans(streamflow['fcst'],streamflow['obs'],s)
    properscoring_crps.append(np.mean(properscoring.crps_ensemble(obs,fcst)) )    

CPU times: user 64.5 ms, sys: 8.31 ms, total: 72.9 ms
Wall time: 70.7 ms


### 5. crps_from_empirical_cdf

In [134]:
%%time
ecdf_crps = []
for s in range(nsite):
    fcst,obs = remove_nans(streamflow['fcst'],streamflow['obs'],s)
    ecdf_crps.append(np.mean(crps_from_empirical_cdf(fcst.T,obs,dim=0))) 

CPU times: user 88.1 ms, sys: 6.94 ms, total: 95.1 ms
Wall time: 91.6 ms


### 6. energy score

In [228]:
%%time
eng_crps = []
for s in range(nsite):
    fcst,obs = remove_nans(streamflow['fcst'],streamflow['obs'],s)
    eng_crps.append(energy_score_multiple(fcst,obs)) 

CPU times: user 1.28 s, sys: 20.9 ms, total: 1.3 s
Wall time: 1.3 s


### make dataframe

In [229]:
df = pd.DataFrame(scores_crps.values,columns = ['scores.crps( 734 ms)'])
df['scoringrules_crps(245 ms)'] = scoringrules_crps
df['properscoring.crps_ensemble(70.7 ms)'] = properscoring_crps
df['vrf_scores_crps(4.58 s)']  = vrf_scores_crps
df['ecdf_crps(91.6 ms)'] = ecdf_crps
df['eng_crps(1.3 s)'] = eng_crps
df.index.name = 'sites'
df

,scores.crps( 734 ms),scoringrules_crps(245 ms),properscoring.crps_ensemble(70.7 ms),vrf_scores_crps(4.58 s),ecdf_crps(91.6 ms),eng_crps(1.3 s)
sites,,,,,,
0,0.665885,0.665885,0.665885,0.665885,0.665885,0.665885
1,5.458856,5.458856,5.458856,5.458856,5.458857,5.458856
2,1.114985,1.114985,1.114985,1.114985,1.114985,1.114985
3,0.531385,0.531384,0.531384,0.531384,0.531384,0.531384
4,3.184142,3.184142,3.184142,3.184142,3.184142,3.184142
5,2.375742,2.375742,2.375742,2.375742,2.375742,2.375742
6,2.858582,2.858577,2.858577,2.858577,2.858577,2.858577
7,13.331986,13.331985,13.331986,13.331986,13.331985,13.331986
8,36.138537,36.138538,36.138535,36.138535,36.138535,36.138535


INFO:distributed.core:Received 'close-stream' from tcp://10.150.201.17:38802; closing.
INFO:distributed.scheduler:Remove worker <WorkerState 'tcp://10.150.201.17:38043', name: SLURMCluster-3-0, status: closing, memory: 0, processing: 0>
INFO:distributed.core:Removing comms to tcp://10.150.201.17:38043
INFO:distributed.core:Received 'close-stream' from tcp://10.150.201.17:38790; closing.
INFO:distributed.scheduler:Remove worker <WorkerState 'tcp://10.150.201.17:46365', name: SLURMCluster-3-1, status: closing, memory: 0, processing: 0>
INFO:distributed.core:Removing comms to tcp://10.150.201.17:46365


In [155]:
num_ens = 500
 
ens1, obs1 = np.random.normal(200,20,num_ens), 150.
ens2, obs2 = np.random.normal(150,30,num_ens), 170.
 
forecasts = [ens1, ens2]
obs = [obs1, obs2]
 
#crps_scores = [crps_from_empirical_cdf(e, o) for e,o in zip(forecasts,obs)]
energy_scores = [energy_score(np.array([e]), np.array([o])) for e,o in zip(forecasts,obs)]
for e,o in zip(forecasts,obs):
    energy_scores1 = energy_score(np.array([e]), np.array([o]))
#print('crps_ecdf:       ', crps_scores, '  mean:', np.mean(crps_scores))
print('optimised energy:', energy_scores, '  mean:', np.mean(energy_scores))

optimised energy: [38.51115277021422, 11.656829032813004]   mean: 25.08399090151361


In [161]:
o

170.0

In [164]:
np.array([e]).shape

(1, 500)

In [166]:
np.array([o]).shape

(1,)

In [223]:
def energy_score_multiple(f_ens, o,return_mean=True):
    
    #num_samples = f_ens.shape[1]
    shape = f_ens.shape
    # handle nans
    mean_fcst = np.mean(f_ens,axis=1) #update by Durga
    nanindx = ~np.isnan(o) & ~np.isnan(mean_fcst)    
    #print(nanindx)
    o = o[nanindx]
    f_ens = f_ens[nanindx,:]    
    if len(shape) == 2 and shape[0] > 1 and shape[1] > 1:  # check for matrix
        num_events = f_ens.shape[0]
        num_ens = f_ens.shape[1]
        crps = []
        for i in range(num_events):
            #ff = f_ens[i,:].reshape(1,num_ens)
            ff = f_ens[i,:]
            ff = ff[np.newaxis, :]   # should be size of 1 by num_samples  
            oo = o[i]
            oo = np.array([oo])
            crps.append(energy_score(ff, oo))   # obs should be 
        #print(f"Fcst shape {ff.shape}, Obs shape: {oo.shape}")
        if return_mean:
            return np.mean(crps)
        else:
            return np.asarray(crps)
    else: 
        return energy_score(f_ens, o)  

In [224]:
energy_score_multiple(np.array([e]), np.array([o]))

11.656829032813004

In [225]:
s=0
fcst,obs = remove_nans(streamflow['fcst'],streamflow['obs'],s)
energy_score_multiple(fcst, obs)
#eng_crps.append(energy_score_multiple(fcst,obs)) 

Fcst shape (1, 50), Obs shape: (1,)


0.665884586651032

In [174]:
fcst.shape

(3740, 50)

In [173]:
obs

array([ 0.888,  0.794,  0.827, ...,  0.673,  0.749, 35.418], dtype=float32)

In [187]:
fcst[1,:][np.newaxis,:].shape

(1, 50)

In [189]:
obs[1]

0.794

In [199]:
obs[1]

0.794

In [212]:
energy_score_multiple(fcst[1,:][np.newaxis,:], np.array([obs[1]]))

0.42821098632812504

In [218]:
energy_score_multiple(fcst, obs)

0.665884586651032

In [214]:
fcst[np.newaxis,:].shape

(1, 3740, 50)

In [182]:
fcst[1,:].shape

(50,)

In [184]:
np.array([e]).shape

(1, 500)

In [215]:
fcst.shape

(3740, 50)